# How to use rugo with JSON Lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mabel-dev/rugo.dev/blob/main/docs/notebooks/jsonl.ipynb)

Read a JSON Lines file (one JSON object per line), set the types you need, read only the fields and rows you need, load it into pandas, and write JSON Lines - with rugo, which needs no PyArrow, pandas or NumPy of its own.

Every example uses the same dataset: 4,630 orbital launch attempts, from Sputnik onwards. More at [rugo.dev](https://rugo.dev).

In [ ]:
%pip install -q rugo pandas

In [ ]:
import os
import urllib.request

if not os.path.exists("space_missions.jsonl"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/rugo.dev/main/docs/data/space_missions.jsonl", "space_missions.jsonl")
print(f"space_missions.jsonl: {os.path.getsize('space_missions.jsonl'):,} bytes")

## Read a file

`read_jsonl` is a context manager that yields a **morsel** - a batch of typed columns. Types are inferred from the first few rows.

In [ ]:
from rugo import jsonl

with jsonl.read_jsonl("space_missions.jsonl") as reader:
    for morsel in reader:
        print(morsel)

## Look at the schema

`read_metadata` counts the lines and infers each field's type from the first few rows.

In [ ]:
meta = jsonl.read_metadata("space_missions.jsonl")
print(meta.num_rows, "rows")
for column in meta.schema_columns:
    print(column)

## Declare the types you know

The first launches have no price, so a small sample sees `Price` as text. `explicit_schema` sets a field's type; the launch time is declared as a timestamp too. (`infer_sample_size=1000` would also find the float.)

In [ ]:
SCHEMA = {"Price": "FLOAT64", "Lauched_at": "TIMESTAMP[us]"}

with jsonl.read_jsonl("space_missions.jsonl", explicit_schema=SCHEMA) as reader:
    morsel = next(iter(reader))

for name, kind in zip(morsel.column_names, morsel.column_types):
    print(f"{name.decode():16} {kind}")

## Read only some fields

Fields you don't name are skipped without being turned into values.

In [ ]:
with jsonl.read_jsonl("space_missions.jsonl", columns=["Company", "Mission"]) as reader:
    for morsel in reader:
        print(morsel)

## Filter rows as you read

`predicates` is a list of `(column, operator, value)`, applied as lines are read. Declare types only for the fields you read.

In [ ]:
with jsonl.read_jsonl(
    "space_missions.jsonl",
    columns=["Company", "Mission", "Price"],
    predicates=[("Company", "==", "SpaceX")],
    explicit_schema={"Price": "FLOAT64"},
) as reader:
    spacex = next(iter(reader))

print(spacex.num_rows, "SpaceX launches")
print(spacex)

## Get Python values

A morsel holds typed columns. `column(name).to_pylist()` turns one into a Python list.

In [ ]:
prices = [p for p in spacex.column("Price").to_pylist() if p is not None]
print("cheapest", min(prices), "- most expensive", max(prices), "($M)")

## Load a file into pandas

Rugo doesn't need pandas, but it's easy to hand data over: each column's `to_pylist()` gives the values, and a dict of lists is a DataFrame. This little helper does it for every morsel a reader yields.

In [ ]:
import pandas as pd


# Every morsel a rugo reader yields, as one pandas DataFrame.
def to_pandas(reader):
    frames = [
        pd.DataFrame({name.decode(): morsel.column(name).to_pylist() for name in morsel.column_names})
        for morsel in reader
    ]
    return pd.concat(frames, ignore_index=True)

In [ ]:
with jsonl.read_jsonl("space_missions.jsonl", explicit_schema=SCHEMA) as reader:
    df = to_pandas(reader)

df.head()

## Write a file

`write_jsonl` turns a morsel into JSON Lines bytes.

In [ ]:
with open("spacex.jsonl", "wb") as f:
    f.write(jsonl.write_jsonl(spacex))

print(open("spacex.jsonl").readline())
print(jsonl.read_metadata("spacex.jsonl").num_rows, "rows written")

## Convert to Parquet

Rugo's readers and writers share the morsel, so converting is a read and a write - with the declared types carried into the Parquet file.

In [ ]:
from rugo import parquet

with jsonl.read_jsonl("space_missions.jsonl", explicit_schema=SCHEMA) as reader:
    with open("space_missions.parquet", "wb") as f:
        with parquet.open_parquet_writer(f.write) as writer:
            for morsel in reader:
                writer.write_row_group(morsel)

for column in parquet.read_metadata("space_missions.parquet").schema_columns:
    print(f"{column.name:16} {column.logical_type}")